# Mega stage evaluation, step 2: replay and figures
Reads the big dumps of `Mega_Dump.ipynb` (143 queries per photo by default) and scores any choice of queries, bar, DINO weight, M and decision without the GPU. Figures and tables go to `results/figures/`.
Reading rules written before the results: (1) any "best subset of queries" is chosen on one half of the photos and scored on the other (2-fold, repeated with random splits), never on the photos it was chosen on; (2) a gain counts only if it holds in both folds and in every condition set that is loaded; (3) wrong answers (error > 8 sigma, -20) are shown next to every score.
Env: `SETS` = comma list of dump folders under `results/rich_dump/` (for example `official150_clean,official150_blur`), `REF` = the set the headline figures use (default the first).

In [ ]:
import os, sys, math, itertools
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
from train.checks import replay
RES = ROOT / "tests/Test_Iteration_12/results"; FIG = RES / "figures"; FIG.mkdir(parents=True, exist_ok=True)
DUMPS = Path(os.environ.get("DUMP_ROOT", str(RES / "rich_dump")))
SET_NAMES = [s for s in os.environ.get("SETS", "").split(",") if s]
assert SET_NAMES, "set SETS to one or more dump folder names"
REF = os.environ.get("REF", SET_NAMES[0])
BAR, GAMMA = 8, 1.0
sets = {name: replay.load_set(DUMPS / name) for name in SET_NAMES}
for name, s in sets.items():
    first = next(iter(s.values()))
    print(name, len(s), "photos,", len(first["queries"]), "queries per photo")
GRID = [(round(q["heading"], 3), round(q["scale"], 3)) for q in next(iter(sets[REF].values()))["queries"]]
HEADINGS = sorted({h for h, s in GRID}); SCALES = sorted({s for h, s in GRID})
print("headings", HEADINGS); print("scales", SCALES)
STANDARD = [GRID.index(k) for k in replay.STANDARD if k in GRID]
print("standard 6 present:", len(STANDARD) == 6)

## Block 1: per-photo, per-query table (the one thing every later block reads)
For each photo and each query: the best candidate by `inliers + gamma * z` (the pipeline's own choice inside the query), its inliers, where it is (x, y), its error in sigma and the points it would earn if answered. Choosing a set of queries = taking the candidate with the best score among them, exactly like the pooled decision. M (how many DINO-ordered candidates go to the fine stage) is applied by zeroing the inliers of later candidates.

In [ ]:
def build(photos, gamma=GAMMA, m=10):
    ids = sorted(photos)
    Q = len(next(iter(photos.values()))["queries"])
    out = {k: np.zeros((len(ids), Q)) for k in ("score", "inl", "x", "y", "err", "pts", "sec", "coarse_hit", "any_inl")}
    for r, pid in enumerate(ids):
        photo = photos[pid]
        for j, q in enumerate(photo["queries"]):
            inl = np.asarray(q["inliers"], float).copy(); inl[m:] = 0.0
            z = np.asarray(q["z"], float); sh = np.asarray(q["shifts"], float)
            usable = inl > 0
            out["sec"][r, j] = sum(q["seconds"].values())
            if not usable.any():
                out["score"][r, j] = -1e9; out["x"][r, j] = q["peaks"][0, 1]; out["y"][r, j] = q["peaks"][0, 2]
                err = replay.error_sigma(photo, out["x"][r, j], out["y"][r, j]); out["err"][r, j] = err
                continue
            s = np.where(usable, inl + gamma * z, -1e9); k = int(np.lexsort((-q["peaks"][:, 0], -s))[0])
            x, y = q["peaks"][k, 1] + sh[k, 0], q["peaks"][k, 2] + sh[k, 1]
            err = replay.error_sigma(photo, x, y)
            out["score"][r, j] = s[k]; out["inl"][r, j] = inl[k]; out["x"][r, j] = x; out["y"][r, j] = y; out["err"][r, j] = err
            out["any_inl"][r, j] = 1.0
        for j in range(Q):
            out["pts"][r, j] = replay.points(out["err"][r, j])
    out["ids"] = ids; out["height"] = np.array([photos[p]["height"] for p in ids])
    return out

def decide(T, subset, bar=BAR):
    """Pooled decision on the queries in `subset`: per photo the best-scoring candidate, answered when its inliers reach the bar. Returns per-photo points, wrong flag, answered flag and the seconds."""
    subset = list(subset)
    sc = T["score"][:, subset]; best = sc.argmax(1); rows = np.arange(len(best)); col = np.array(subset)[best]
    inl = T["inl"][rows, col]; err = T["err"][rows, col]
    answered = inl >= bar
    pts = np.where(answered, T["pts"][rows, col], 0.0)
    wrong = answered & (err > 8)
    return dict(points=pts, answered=answered, wrong=wrong, err=err, inl=inl, seconds=T["sec"][:, subset].sum(1))

def summary(d):
    n = len(d["points"])
    return dict(score60=d["points"].mean() * 6, answered=d["answered"].mean(), wrong=int(d["wrong"].sum()), sec=d["seconds"].mean())

TABLES = {name: build(s) for name, s in sets.items()}
T = TABLES[REF]
ALL = list(range(T["score"].shape[1]))
rows = [dict(set=n, **{k: round(v, 3) for k, v in summary(decide(t, STANDARD)).items()}) for n, t in TABLES.items()]
display(pd.DataFrame(rows).rename(columns=dict(sec="sec (sum of the 6 queries)")))

## Figure 1: which heading and scale find the photo?
Two maps per condition. Left: share of photos whose best candidate of that ONE query is within 2 sigma of the truth (the query could answer it). Right: the same, but only when the candidate also reaches the bar (the query WOULD answer it). Black frames mark the 6 standard queries.

In [ ]:
def grid_map(values_by_query):
    img = np.full((len(SCALES), len(HEADINGS)), np.nan)
    for j, (h, s) in enumerate(GRID):
        img[SCALES.index(s), HEADINGS.index(h)] = values_by_query[j]
    return img

def draw_maps(name, Tn):
    find = (Tn["err"] <= 2.0).mean(0); answer = ((Tn["err"] <= 2.0) & (Tn["inl"] >= BAR)).mean(0)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    for ax, vals, title in zip(axes, (find, answer), ("best candidate within 2 sigma", "...and inliers >= bar %d" % BAR)):
        im = ax.imshow(grid_map(vals), origin="lower", aspect="auto", cmap="viridis", vmin=0)
        ax.set_xticks(range(len(HEADINGS))); ax.set_xticklabels(["%g" % h for h in HEADINGS], rotation=60); ax.set_yticks(range(len(SCALES))); ax.set_yticklabels(["%.2f" % s for s in SCALES])
        ax.set_xlabel("heading offset (deg)"); ax.set_ylabel("scale factor"); ax.set_title("%s: %s" % (name, title))
        for j in STANDARD:
            h, s = GRID[j]; ax.add_patch(plt.Rectangle((HEADINGS.index(h) - .5, SCALES.index(s) - .5), 1, 1, fill=False, ec="k", lw=1.6))
        plt.colorbar(im, ax=ax, label="share of photos")
    fig.tight_layout(); fig.savefig(FIG / ("fig1_query_maps_%s.png" % name), dpi=130); plt.show(); plt.close(fig)

for name, Tn in TABLES.items():
    draw_maps(name, Tn)

## Figure 2: how many queries are worth running, and which?
Greedy forward selection of queries: start empty, add the query that raises the pooled score most. Chosen on one half of the photos, scored on the other half (100 random 50/50 splits, mean and 10-90 % band); the curve of RANDOM query sets of the same size is the control, and the standard 6, the research 27-style set and all queries are marked. The x axis is the number of queries and, below, the seconds per photo that many queries cost on this GPU.

In [ ]:
def pooled_points(Tn, subset, rows, bar=BAR):
    sub = np.array(subset); sc = Tn["score"][np.ix_(rows, sub)]; best = sc.argmax(1); col = sub[best]; r = np.array(rows)
    inl = Tn["inl"][r, col]; ans = inl >= bar
    return np.where(ans, Tn["pts"][r, col], 0.0), ans & (Tn["err"][r, col] > 8)

def greedy(Tn, rows, steps, bar=BAR):
    chosen = []; curve = []
    for _ in range(steps):
        best_q, best_v = None, -1e9
        for q in range(Tn["score"].shape[1]):
            if q in chosen:
                continue
            pts, wrong = pooled_points(Tn, chosen + [q], rows, bar)
            v = pts.sum() - 20 * 0.0
            if v > best_v:
                best_q, best_v = q, v
        chosen.append(best_q); curve.append(best_v / len(rows) * 6)
    return chosen, curve

STEPS = min(40, len(ALL))
rng = np.random.default_rng(0)
cv = {n: [] for n in range(1, STEPS + 1)}; rand = {n: [] for n in range(1, STEPS + 1)}; picked = []
n_photos = len(T["ids"])
for split in range(30):
    order = rng.permutation(n_photos); a, b = order[: n_photos // 2], order[n_photos // 2:]
    for train_rows, test_rows in ((a, b), (b, a)):
        chosen, _ = greedy(T, list(train_rows), STEPS); picked.append(chosen)
        for n in range(1, STEPS + 1):
            pts, wrong = pooled_points(T, chosen[:n], list(test_rows)); cv[n].append(pts.mean() * 6)
            rq = list(rng.choice(len(ALL), n, replace=False)); pts, wrong = pooled_points(T, rq, list(test_rows)); rand[n].append(pts.mean() * 6)
sec_per_query = T["sec"].mean()
xs = np.arange(1, STEPS + 1)
fig, ax = plt.subplots(figsize=(9, 5))
for data, label, color in ((cv, "greedy (chosen on the other half)", "C0"), (rand, "random sets of the same size", "C1")):
    mean = np.array([np.mean(data[n]) for n in xs]); lo = np.array([np.percentile(data[n], 10) for n in xs]); hi = np.array([np.percentile(data[n], 90) for n in xs])
    ax.plot(xs, mean, color=color, label=label); ax.fill_between(xs, lo, hi, color=color, alpha=.2)
std_pts, _ = pooled_points(T, STANDARD, list(range(n_photos))); all_pts, _ = pooled_points(T, ALL, list(range(n_photos)))
ax.axhline(std_pts.mean() * 6, color="k", ls="--", label="standard 6 queries (%.1f)" % (std_pts.mean() * 6)); ax.axhline(all_pts.mean() * 6, color="g", ls=":", label="all %d queries (%.1f)" % (len(ALL), all_pts.mean() * 6))
ax.set_xlabel("number of queries"); ax.set_ylabel("score out of 60 (held-out half)"); ax.set_title("%s: score against queries, chosen sets against random sets" % REF); ax.legend(loc="lower right"); ax.grid(alpha=.3)
sec = ax.secondary_xaxis("top", functions=(lambda n: n * sec_per_query, lambda s: s / sec_per_query)); sec.set_xlabel("seconds per photo (this GPU, no copies)")
fig.tight_layout(); fig.savefig(FIG / "fig2_greedy_vs_random.png", dpi=130); plt.show(); plt.close(fig)
count = pd.Series([q for chosen in picked for q in chosen[:8]]).value_counts().head(12)
display(pd.DataFrame({"query (heading, scale)": [GRID[q] for q in count.index], "chosen in first 8 (of %d runs)" % len(picked): count.values}))
pd.DataFrame(dict(n_queries=xs, greedy_cv=[np.mean(cv[n]) for n in xs], random=[np.mean(rand[n]) for n in xs])).to_csv(RES / ("fig2_greedy_vs_random_%s.csv" % REF), index=False)

## Figure 3: the answer bar, fine steps
Pooled decision of the standard 6, of the 18 and of all queries, bar 3 to 30 step 1: score, answered share and wrong answers. The bar is the one parameter the submission cannot test on the hidden photos, so the curve must be flat near the chosen value to be safe.

In [ ]:
BARS = list(range(3, 31))
subsets = {"standard 6": STANDARD, "all %d" % len(ALL): ALL}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4)); records = []
for name, Tn in TABLES.items():
    for sname, subset in subsets.items():
        sc, an, wr = [], [], []
        for bar in BARS:
            d = decide(Tn, subset, bar); s = summary(d); sc.append(s["score60"]); an.append(s["answered"]); wr.append(s["wrong"])
            records.append(dict(set=name, subset=sname, bar=bar, **s))
        style = "-" if sname.startswith("standard") else "--"
        axes[0].plot(BARS, sc, style, label="%s %s" % (name, sname)); axes[1].plot(BARS, an, style); axes[2].plot(BARS, wr, style)
axes[0].set_ylabel("score /60"); axes[1].set_ylabel("answered share"); axes[2].set_ylabel("wrong answers (count)")
for ax in axes:
    ax.set_xlabel("bar (inliers)"); ax.axvline(BAR, color="k", lw=.6); ax.grid(alpha=.3)
axes[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(FIG / "fig3_bar_sweep.png", dpi=130); plt.show(); plt.close(fig)
pd.DataFrame(records).to_csv(RES / "fig3_bar_sweep.csv", index=False)

## Figure 4: DINO weight and M (how many candidates reach the fine matcher)
Gamma is the weight of the DINO z-score in the choice `inliers + gamma * z`; M is how many of the DINO-ordered candidates of each query are matched. Both from the stored numbers (candidates beyond M get 0 inliers).

In [ ]:
GAMMAS = [0, 0.25, 0.5, 0.75, 1, 1.5, 2, 3]; MS = [1, 2, 3, 5, 7, 10]
rows = []
for name, s in sets.items():
    for g in GAMMAS:
        Tn = build(s, gamma=g); d = decide(Tn, STANDARD); rows.append(dict(set=name, kind="gamma", value=g, **summary(d)))
    for m in MS:
        Tn = build(s, m=m); d = decide(Tn, STANDARD); rows.append(dict(set=name, kind="M", value=m, **summary(d)))
R4 = pd.DataFrame(rows); R4.to_csv(RES / "fig4_gamma_M.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for name in sets:
    for ax, kind in zip(axes, ("gamma", "M")):
        part = R4[(R4.set == name) & (R4.kind == kind)]; ax.plot(part.value, part.score60, "o-", label=name)
axes[0].set_xlabel("DINO weight gamma"); axes[1].set_xlabel("M fine candidates per query")
for ax in axes:
    ax.set_ylabel("score /60 (standard 6 queries)"); ax.grid(alpha=.3)
axes[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(FIG / "fig4_gamma_M.png", dpi=130); plt.show(); plt.close(fig)
display(R4.pivot_table(index=["kind", "value"], columns="set", values="score60").round(2))

## Figure 5: where the points go, by flight height
Score of the standard 6, the held-out greedy set (the 8 most chosen queries) and all queries per height band; and the share of photos that no query of the whole grid can answer (the ceiling the grid sets for any query-based method).

In [ ]:
best8 = [q for q in pd.Series([q for chosen in picked for q in chosen[:8]]).value_counts().head(8).index]
bands = pd.qcut(T["height"], 5, duplicates="drop"); labels = [str(b) for b in bands.categories]
rows = []
for sname, subset in (("standard 6", STANDARD), ("most-chosen 8", best8), ("all %d" % len(ALL), ALL)):
    d = decide(T, subset)
    for b in bands.categories:
        mask = np.asarray(bands == b); rows.append(dict(subset=sname, band=str(b), score60=d["points"][mask].mean() * 6, n=int(mask.sum())))
ceiling = ((T["err"] <= 2.0) & (T["inl"] >= BAR)).any(1)
for b in bands.categories:
    mask = np.asarray(bands == b); rows.append(dict(subset="ceiling (some query answers right)", band=str(b), score60=ceiling[mask].mean() * 60, n=int(mask.sum())))
R5 = pd.DataFrame(rows); R5.to_csv(RES / ("fig5_height_%s.csv" % REF), index=False)
piv = R5.pivot(index="band", columns="subset", values="score60"); piv = piv.loc[[str(b) for b in bands.categories]]
ax = piv.plot(kind="bar", figsize=(10, 4.5)); ax.set_ylabel("score /60 within the band (ceiling: share x 60)"); ax.set_title("%s: score by flight height band" % REF); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "fig5_height_bands.png", dpi=130); plt.show(); plt.close()
display(piv.round(1))

## Figure 6: robustness across conditions
Score and wrong answers of the standard 6, the held-out greedy 8 and all queries in every loaded condition (clean, blur, invert, ...). A query set is only better if it is better in every condition, not on average.

In [ ]:
rows = []
for name, Tn in TABLES.items():
    for sname, subset in (("standard 6", STANDARD), ("most-chosen 8", best8), ("all %d" % len(ALL), ALL)):
        d = decide(Tn, subset); rows.append(dict(set=name, subset=sname, **summary(d)))
R6 = pd.DataFrame(rows); R6.to_csv(RES / "fig6_conditions.csv", index=False)
piv = R6.pivot(index="set", columns="subset", values="score60")
ax = piv.plot(kind="bar", figsize=(10, 4.5)); ax.set_ylabel("score /60"); ax.set_title("score by condition and query set (best-8 chosen on the reference set: in-sample for it)"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "fig6_conditions.png", dpi=130); plt.show(); plt.close()
display(R6.round(2))

## Figure 7 (organiser feedback, geometry): is the fitted scale / heading a stand-in for a geometry error?
For every photo that some query answers right (error <= 2 sigma, inliers >= bar) the query with the most inliers gives the BEST scale and heading offset for that photo. If the pipeline's constants (yaw offset 180, scale 1.04, roll bias) were exact, the best scale would sit at one value and not move with height, pitch or roll. Plotted against flight height, pitch and roll, per condition; the median per bin is the line. A trend with height or tilt = a geometry term still missing.

In [ ]:
def best_query_table(name):
    Tn = TABLES[name]; photos = sets[name]; rows = []
    ok = (Tn["err"] <= 2.0) & (Tn["inl"] >= BAR)
    for r, pid in enumerate(Tn["ids"]):
        if not ok[r].any():
            continue
        j = int(np.where(ok[r], Tn["inl"][r], -1).argmax()); h, s = GRID[j]
        rows.append(dict(photo=pid, height=Tn["height"][r], pitch=photos[pid]["pitch"], roll=photos[pid]["roll"], heading_off=h, scale=s, inl=Tn["inl"][r, j]))
    return pd.DataFrame(rows)

fig, axes = plt.subplots(2, 3, figsize=(15, 7.5)); tables7 = {}
for name in TABLES:
    D = best_query_table(name); tables7[name] = D; D.to_csv(RES / ("fig7_best_query_%s.csv" % name), index=False)
    for col, x in enumerate(("height", "pitch", "roll")):
        for row, y in enumerate(("scale", "heading_off")):
            ax = axes[row, col]; ax.scatter(D[x], D[y] + np.random.default_rng(1).normal(0, 0.004 if y == "scale" else 0.2, len(D)), s=8, alpha=.35, label=name)
            if len(D) > 20 and D[x].nunique() > 3:
                bins = pd.qcut(D[x], min(6, D[x].nunique()), duplicates="drop"); med = D.groupby(bins, observed=True).agg(x=(x, "median"), y=(y, "median"))
                ax.plot(med.x, med.y, "o-", lw=2)
            ax.set_xlabel(x); ax.set_ylabel("best " + y); ax.grid(alpha=.3)
axes[0, 0].legend(fontsize=7); fig.suptitle("best scale and heading offset of each photo against height, pitch, roll"); fig.tight_layout(); fig.savefig(FIG / "fig7_best_scale_heading.png", dpi=130); plt.show(); plt.close(fig)
display(pd.concat({n: d[["scale", "heading_off"]].describe().T for n, d in tables7.items()}).round(3))

## Figure 8 (organiser feedback, precision): signed error against height, pitch and roll
For the photos the standard 6 queries answer, the signed position error (east, north, metres) of the answer against the truth, against flight height, pitch and roll. A residual that moves with one of them is a geometry term the constants are absorbing. The line is the median per bin; the band is the 25-75 % range.

In [ ]:
def signed_errors(name, subset=STANDARD):
    Tn = TABLES[name]; photos = sets[name]; d = decide(Tn, subset); rows = []
    sub = np.array(subset); best = Tn["score"][:, sub].argmax(1); col = sub[best]
    for r, pid in enumerate(Tn["ids"]):
        if not d["answered"][r]:
            continue
        x, y = Tn["x"][r, col[r]], Tn["y"][r, col[r]]; tx, ty = photos[pid]["truth"]
        rows.append(dict(photo=pid, height=Tn["height"][r], pitch=photos[pid]["pitch"], roll=photos[pid]["roll"], east=x - tx, north=y - ty, err_sigma=Tn["err"][r, col[r]], sigma=photos[pid]["sigma"]))
    return pd.DataFrame(rows)

fig, axes = plt.subplots(2, 3, figsize=(15, 7.5)); rows = []
for name in TABLES:
    D = signed_errors(name); D.to_csv(RES / ("fig8_signed_error_%s.csv" % name), index=False)
    D = D[D.err_sigma <= 8]
    for col, x in enumerate(("height", "pitch", "roll")):
        for row, y in enumerate(("east", "north")):
            ax = axes[row, col]; ax.scatter(D[x], D[y], s=8, alpha=.3, label=name)
            if len(D) > 20 and D[x].nunique() > 3:
                bins = pd.qcut(D[x], min(6, D[x].nunique()), duplicates="drop"); g = D.groupby(bins, observed=True)
                xs_, med, lo, hi = g[x].median(), g[y].median(), g[y].quantile(.25), g[y].quantile(.75)
                ax.plot(xs_, med, "o-", lw=2); ax.fill_between(xs_, lo, hi, alpha=.15)
                rows.append(dict(set=name, x=x, y=y, slope_m_per_unit=float(np.polyfit(D[x], D[y], 1)[0]), corr=float(np.corrcoef(D[x], D[y])[0, 1])))
            ax.axhline(0, color="k", lw=.6); ax.set_xlabel(x); ax.set_ylabel(y + " error (m)"); ax.grid(alpha=.3)
axes[0, 0].legend(fontsize=7); fig.suptitle("signed error of answered photos (<= 8 sigma) against height, pitch, roll"); fig.tight_layout(); fig.savefig(FIG / "fig8_signed_error.png", dpi=130); plt.show(); plt.close(fig)
display(pd.DataFrame(rows).round(3))

## Figure 9 (organiser feedback, the clock): how many seconds does a photo need, and what does the time guard cost?
Sequential replay: queries run one after another; the photo is answered as soon as the best candidate of the queries run so far reaches the bar (a simplification of the real decision, same for every order). Orders compared: the grid as written (standard 6 first, then the rest by heading), standard 6 then the rest ordered by the first pass's best heading (the organisers' suggestion), and an oracle (best query first). Seconds are the dump's own, multiplied by `JETSON_X` (the submitted code took about 13 s per photo on the Jetson against 4-5 s on the laptop, so about 3, set it by env). Score and answered share against the time guard.

In [ ]:
JETSON_X = float(os.environ.get("JETSON_X", "3.0")); GUARDS = [8, 10, 12, 15, 18, 22, 26, 30, 40]
def first_pass_best_heading(Tn, r):
    sc = Tn["score"][r, STANDARD]; return GRID[STANDARD[int(sc.argmax())]][0]
def run_order(Tn, r, order, guard):
    best_s, best_q, used = -1e9, None, 0.0
    for q in order:
        cost = Tn["sec"][r, q] * JETSON_X
        if used + cost > guard and best_q is not None:
            break
        used += cost
        if Tn["score"][r, q] > best_s:
            best_s, best_q = Tn["score"][r, q], q
        if best_q is not None and Tn["inl"][r, best_q] >= BAR:
            return best_q, used
    return best_q, used
rest = [j for j in range(len(GRID)) if j not in STANDARD]
rows = []
for name, Tn in TABLES.items():
    for r in range(len(Tn["ids"])):
        h_star = first_pass_best_heading(Tn, r)
        orders = {"grid order": STANDARD + sorted(rest, key=lambda j: (abs(GRID[j][0]), GRID[j][1])),
                  "by first-pass heading": STANDARD + sorted(rest, key=lambda j: (abs(GRID[j][0] - h_star), abs(GRID[j][1] - 1.0))),
                  "oracle (best query first)": [int(np.where(Tn["inl"][r] >= BAR, Tn["pts"][r], -1).argmax())] + STANDARD + rest}
        for oname, order in orders.items():
            for guard in GUARDS:
                q, used = run_order(Tn, r, order, guard)
                ans = q is not None and Tn["inl"][r, q] >= BAR
                rows.append(dict(set=name, order=oname, guard=guard, photo=r, answered=ans, points=Tn["pts"][r, q] if ans else 0.0, seconds=used, wrong=bool(ans and Tn["err"][r, q] > 8)))
R9 = pd.DataFrame(rows).groupby(["set", "order", "guard"]).agg(score60=("points", lambda v: v.mean() * 6), answered=("answered", "mean"), wrong=("wrong", "sum"), seconds=("seconds", "mean")).reset_index()
R9.to_csv(RES / "fig9_time_guard.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for (name, oname), part in R9.groupby(["set", "order"]):
    axes[0].plot(part.guard, part.score60, "o-", label="%s, %s" % (name, oname)); axes[1].plot(part.guard, part.answered, "o-"); axes[2].plot(part.guard, part.seconds, "o-")
for ax, yl in zip(axes, ("score /60", "answered share", "mean seconds used (Jetson-scaled)")):
    ax.set_xlabel("time guard (s)"); ax.set_ylabel(yl); ax.axvline(22, color="k", lw=.6); ax.grid(alpha=.3)
axes[0].legend(fontsize=6); fig.suptitle("time guard against score, JETSON_X = %g (black line: the submitted 22 s)" % JETSON_X); fig.tight_layout(); fig.savefig(FIG / "fig9_time_guard.png", dpi=130); plt.show(); plt.close(fig)
display(R9[R9.guard.isin([15, 22, 26, 30])].round(2))

## Figure 10: the pipeline's real decision, replayed on the stored queries
`replay.decide_current` is the submission decision on the stored matches: the 6 standard queries pooled, the bar, the agreement rule (two queries within 1 sigma, each with at least 3 inliers), then the 12-query second pass. Grid keys outside the dump are skipped, so the set needs the standard 6 and the 12 wide queries (headings -10 to 10 x scales 0.95 / 1.0 / 1.05), which the default grid contains. Compared per condition with the bar-only decision of Figure 3; bars 6, 8, 10 and the agreement rule on or off (agree_min = 99 switches it off).

In [ ]:
need = set(replay.STANDARD) | set(replay.WIDE)
have = {(round(h, 3), round(s, 3)) for h, s in GRID}
print("grid has all standard + wide queries:", need <= have)
rows = []
for name, photos in sets.items():
    for bar in (6, 8, 10):
        for agree_min, label in ((2, "agreement on"), (99, "agreement off")):
            for escalate in (False, True):
                pts, wrong, answered, secs = [], 0, 0, []
                for pid, photo in photos.items():
                    r = replay.decide_current(photo, escalate=escalate, bar=bar, agree_min=agree_min)
                    pts.append(r["points"]); wrong += int(r["answered"] and r["err_sigma"] > 8); answered += int(r["answered"]); secs.append(r["seconds"])
                rows.append(dict(set=name, bar=bar, agreement=label, escalate=escalate, score60=np.mean(pts) * 6, answered=answered / len(pts), wrong=wrong, seconds=np.mean(secs)))
R10 = pd.DataFrame(rows); R10.to_csv(RES / "fig10_real_decision.csv", index=False)
piv = R10[(R10.bar == 8)].pivot_table(index=["agreement", "escalate"], columns="set", values="score60").round(2)
display(piv)
fig, axes = plt.subplots(1, len(sets), figsize=(5.5 * len(sets), 4.2), squeeze=False)
for ax, name in zip(axes[0], sets):
    part = R10[R10.set == name]
    for (agr, esc), g in part.groupby(["agreement", "escalate"]):
        ax.plot(g.bar, g.score60, "o-", label="%s, %s" % (agr, "second pass" if esc else "first pass only"))
    ax.set_title(name); ax.set_xlabel("bar"); ax.set_ylabel("score /60"); ax.grid(alpha=.3)
axes[0][0].legend(fontsize=7); fig.tight_layout(); fig.savefig(FIG / "fig10_real_decision.png", dpi=130); plt.show(); plt.close(fig)
display(R10[(R10.bar == 8) & (R10.agreement == "agreement on")].round(2))